In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

In [ ]:
matches.head()

In [ ]:
deliveries.head()

In [ ]:
matches = matches.rename(columns={'id': 'match_id'})

In [ ]:
df = deliveries.merge(matches, on='match_id')

In [ ]:
df = df[df['inning'] == 2]

In [ ]:
total_score_df = deliveries.groupby(['match_id','inning']).sum()['total_runs'].reset_index()

# get 1st innings score
first_innings = total_score_df[total_score_df['inning'] == 1]

# rename
first_innings = first_innings.rename(columns={'total_runs':'target'})

# merge with main df
df = df.merge(first_innings[['match_id','target']], on='match_id')

# target = +1 run
df['target'] = df['target'] + 1

In [ ]:
df['current_score'] = df.groupby('match_id')['total_runs'].cumsum()

In [ ]:
df['ball_number'] = df['over']*6 + df['ball']

df['balls_remaining'] = 120 - df['ball_number']

In [ ]:
df['player_dismissed'] = df['player_dismissed'].fillna(0)

df['is_wicket'] = df['player_dismissed'].apply(lambda x: 0 if x == 0 else 1)

df['wickets_fallen'] = df.groupby('match_id')['is_wicket'].cumsum()

df['wickets_remaining'] = 10 - df['wickets_fallen']

In [ ]:
df['current_run_rate'] = df['current_score'] / (df['ball_number']/6)

df['runs_remaining'] = df['target'] - df['current_score']

df['required_run_rate'] = df['runs_remaining'] / (df['balls_remaining']/6)

In [ ]:
# winner column already in matches
df['batting_team'] = df['batting_team']

df['result'] = df.apply(
    lambda row: 1 if row['batting_team'] == row['winner'] else 0,
    axis=1
)

In [ ]:
df = df[df['balls_remaining'] > 0]

In [ ]:
final_df = df[[
    'balls_remaining',
    'wickets_remaining',
    'current_run_rate',
    'required_run_rate',
    'result'
]]

final_df = final_df.dropna()

In [ ]:
import numpy as np

final_df = final_df.replace([np.inf, -np.inf], np.nan)
final_df = final_df.dropna()

In [ ]:
print(final_df.isnull().sum())
print(np.isinf(final_df).sum())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = final_df.drop('result', axis=1)
y = final_df['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression()
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

In [ ]:
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(X_test)[:,1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    reg_lambda=1,
    reg_alpha=0.5,
    random_state=42
)
xgb.fit(X_train, y_train)

# predictions
y_pred = xgb.predict(X_test)
y_prob = xgb.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
import matplotlib.pyplot as plt

importances = xgb.feature_importances_
features = X.columns

plt.barh(features, importances)
plt.xlabel("Importance")
plt.title("Feature Importance")
plt.show()

In [ ]:
import os
import pickle

os.makedirs("models", exist_ok=True)

pickle.dump(xgb, open("models/xgb_model.pkl", "wb"))

In [ ]:
model = pickle.load(open("models/xgb_model.pkl", "rb"))

model.predict_proba([[30,5,8.5,9.2]])